In [ ]:
%pip install langgraph langchain-openai langchain dotenv arize-phoenix-otel openinference-instrumentation-langchain

In [ ]:
from langgraph.prebuilt import create_react_agent
import dotenv
import uuid
from langchain_openai import ChatOpenAI
import requests



In [ ]:
dotenv.load_dotenv()

In [ ]:
from phoenix.otel import register
from openinference.instrumentation import using_metadata

# configure the Phoenix tracer
tracer_provider = register(
  project_name="pydata-seattle-2025-workshop", # Default is 'default'
  auto_instrument=True # Auto-instrument your app based on installed OI dependencies
)

In [ ]:
user_id = uuid.uuid4()


In [ ]:
print(f"User ID: {user_id}  ")

In [ ]:
default_metadata = {
    "user_id": user_id,
    
}

In [ ]:
print (f"""Add this line to filter traces 

metadata['user_id'] == "{user_id}"

""")

In [ ]:
def get_weather(city: str) -> str:  
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

In [ ]:
agent = create_react_agent(
    model="openai:gpt-4o-mini",   
    tools=[get_weather],  
    prompt="You are a helpful assistant"  
)

In [ ]:
with using_metadata(default_metadata):
    # Run the agent
    res = agent.invoke(
        {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
    )
res

In [ ]:
BASE = "http://34.11.200.211:8080"

llm = ChatOpenAI(
    model="gpt-4o-mini",
    base_url=f"{BASE}/v1",
    temperature=0.2,
    max_tokens=512,
)

llm.invoke("what is the weather in sf")




In [ ]:
def tavily_search(query, **kw):
    r = requests.post(f"{BASE}/v1/tavily/search",
                      json={"query": query, **kw}, timeout=60)
    r.raise_for_status()
    return r.json()

res = tavily_search("What is Langraph?")
print(res)

## References

https://arize.com/docs/phoenix/integrations/python/langgraph/langgraph-tracing